일단 기본 4개 노드 4개 원형 엣지 구성으로 감


In [ ]:
from qiskit import *


import torch

import numpy as np
import matplotlib.pyplot as plt

import random

from EGATE import *
# from NNVQE_HEA import *

In [ ]:
# H-graph latent vector 구성
L = 3
N = L*L

YY_data = 5
ZZ_1_data = 9
ZZ_2_data = 9

num_qubits = 9
batch_size = 5
num_epochs = 51
num_layer = 3

tot_num_train_data = ZZ_1_data * ZZ_2_data*YY_data

latent_data = []
edge_full_data = []
node_full_data = []

if __name__ == "__main__":
    torch.manual_seed(1)
    np.random.seed(1)
    random.seed(1)
    for J_yy in np.linspace(-1.0,1.0, YY_data):
        for J1_zz in np.linspace(-2.0,2.0,ZZ_1_data):
            for J2_zz in np.linspace(-2.0,2.0,ZZ_2_data):
                node_feats = two_dim_lattice_encoding(N)
                node_full_data.append(node_feats)

                edges, edge_feats_dict = generate_hamiltonian_graph_edge_2d(N, L,
                    # 최근접 이웃(interaction strength)
                    J1_xx=1.0, J1_yy=J_yy, J1_zz=J1_zz,
                    # 두 번째 이웃(interaction strength)
                    J2_xx=1.0, J2_yy=J_yy, J2_zz=J2_zz,
                )
                
                # print(edge_feats_dict)
                edge_full_data.append(edge_feats_dict)
                # print("\n=== Training Graph AutoEncoder [EGAT Layer with λ splitting] ===",i)
    model = EGATEAutoEncoder(
        node_in_dim= 2,
        edge_in_dim= 3,
        node_hidden_dim= 2,  
        edge_hidden_dim= 3,
        decoder_hidden_dim = 45,
        num_layers=num_layer,
        lambda_param=0.5,
        latent_dim = 8,
        num_edges=len(edges)
    )    
    latent_data, mse_list, H_rec, E_rec, H_mse_list, E_mse_list = train_multi_graph_minibatch(model, tot_num_train_data, edges, edge_full_data, node_full_data, 
                                                                                              num_epochs=num_epochs,batch_size=batch_size, beta = 10.0)
    torch.save(model.state_dict(), "gae_model.pth")
    for i, lat in enumerate(latent_data):
        # print(edge_full_data[i])
            print(f"Graph {i+1}: latent = {lat}")

In [ ]:
plt.plot(list(range(len(list(mse_list)))), list(mse_list))
plt.show()

In [ ]:
# H_mse_list = [t.detach().cpu().item() for t in H_mse_list]
plt.plot(list(range(len(list(H_mse_list)))), H_mse_list)
plt.show()

In [ ]:
# E_mse_list = [t.detach().cpu().item() for t in E_mse_list]
plt.plot(list(range(len(list(H_mse_list)))), E_mse_list)
plt.show()

In [ ]:
torch.save(edge_full_data, 'edge_full_data_train.pt')
torch.save(latent_data, 'latent_data_train.pt')
torch.save(node_full_data,'node_full_data_train.pt')
# 불러오기


In [ ]:
ZZ_1_test_data = 50
ZZ_2_test_data = 50
YY_test_data = 15
num_test_data = ZZ_1_test_data * ZZ_2_test_data*YY_test_data
test_latent_data = []
test_edge_full_data = []
test_node_full_data = []

In [ ]:

if __name__ == "__main__":
    for J_yy in np.linspace(-3.0,3.0,YY_test_data):
        for J1_zz in np.linspace(-5.0,5.0,ZZ_1_test_data):
            for J2_zz in np.linspace(-5.0,5.0,ZZ_2_test_data):
                node_feats = two_dim_lattice_encoding(N)
                test_node_full_data.append(node_feats)

                edges, edge_feats_dict = generate_hamiltonian_graph_edge_2d(N, L,
                    # 최근접 이웃(interaction strength)
                    J1_xx=1.0, J1_yy=J_yy, J1_zz=J1_zz,
                    # 두 번째 이웃(interaction strength)
                    J2_xx=1.0, J2_yy=J_yy, J2_zz=J2_zz,
                )
                
                # print(edge_feats_dict)
                test_edge_full_data.append(edge_feats_dict)

In [ ]:
model = EGATEAutoEncoder(
        node_in_dim= 2,
        edge_in_dim= 3,
        node_hidden_dim= 2,  
        edge_hidden_dim= 3,
        decoder_hidden_dim = 45,
        num_layers=num_layer,
        lambda_param=0.5,
        latent_dim = 8,
        num_edges=len(edges)
    )  
model.load_state_dict(torch.load("gae_model.pth"))
model.eval()
mse = nn.MSELoss()
mse_list = []

H_rec = []
E_rec = []
for i in range(num_test_data):
    with torch.no_grad():
        node_feats = test_node_full_data[i]
        edge_feats_dict= test_edge_full_data[i]
        E_in_list = [edge_feats_dict[e] for e in edges]
        E_in = torch.stack(E_in_list, dim=0)
    
        H_re, E_re, graph_latent = model(node_feats, E_in, edges)

        test_latent_data.append(graph_latent)
        E_loss = mse(E_re, E_in)
        H_loss = mse(H_re, node_feats)
        mse_list.append(E_loss + H_loss)

In [ ]:
# E_mse_list = [t.detach().cpu().item() for t in E_mse_list]
plt.plot(list( np.linspace(-10.0,10.0,num_test_data)), mse_list)
print(np.mean(mse_list))
plt.show()

In [ ]:
torch.save(test_edge_full_data, 'edge_full_data_test.pt')
torch.save(test_latent_data, 'latent_data_test.pt')
torch.save(test_node_full_data, 'node_full_data_test.pt')

# 불러오기
